# Business Entity Resolution — Complete End-to-End Pipeline (Google Colab)

**Run all cells top to bottom (Runtime ▸ Run all).** This notebook is fully
self-contained: it installs dependencies, fetches the dataset, trains the
LightGBM matcher, generates `matching_results.tsv` + `candidate_pairs.tsv`,
validates them, and saves everything (plus a submission zip) to your Google
Drive.

*Pipeline:* country partitioning → multi-key inverted-index blocking
(top-12 candidates per Source-1 entity) → 18 RapidFuzz/token features →
LightGBM classifier → threshold tuned for macro F₀.₅ (singletons included).


In [ ]:
# STEP 0 — Install dependencies
!pip -q install rapidfuzz lightgbm gdown
import os
for d in ['src', 'utils', 'output', 'models', 'work']:
    os.makedirs(d, exist_ok=True)
print('ready')


In [ ]:
# STEP 1 — Mount Google Drive (used to SAVE results; dataset download
# does not require it, but saving results does)
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# ============================================================
# STEP 2 — Get the dataset
# Option A (default): download straight from the shared Google Drive dataset
#   folder (https://drive.google.com/drive/folders/1kYWqTFLyT-6d5kql3dbubDRIWBy_eek0)
# Option B: if you already have the dataset in YOUR Drive, set
#   DATASET_DIR_IN_DRIVE below (folder containing train/ and test/) and the
#   files are copied from there instead.
# ============================================================
import os, shutil

DATASET_DIR_IN_DRIVE = ""   # e.g. "/content/drive/MyDrive/dataset" (optional)

DATA_DIR = "/content/dataset"
FILE_IDS = {
    "train/train_source1.tsv": "17iDDtk9AzDanNhBTCBuOuEGyuD7Hm1RZ",
    "train/train_source2.tsv": "1mxUeH9v97X9nJpRA-gHFMOjeT5UbwpKK",
    "train/train_source3.tsv": "1mYkKYF07auePjI0X74PxTCqR42Nnua9X",
    "train/train_ground_truth.tsv": "1YEMGKmoXSvLRJJ7e_-fhbfXpgNTb8GwX",
    "test/test_source1.tsv": "1eezHtRDobL9HSSszU_JdiDEgjFN3HLrE",
    "test/test_source2.tsv": "1caUOZ8OPejc8UqlrCX2nJRtXtNOQXfcf",
    "test/test_source3.tsv": "1XRU_mlvUJVCEeUkoDhtC34hRdY79OjdU"
}
EXPECTED = {
    "train/train_source1.tsv": 210069713,
    "train/train_source2.tsv": 489301488,
    "train/train_source3.tsv": 503705637,
    "train/train_ground_truth.tsv": 127015583,
    "test/test_source1.tsv": 175022086,
    "test/test_source2.tsv": 509456422,
    "test/test_source3.tsv": 506002772
}

os.makedirs(DATA_DIR + "/train", exist_ok=True)
os.makedirs(DATA_DIR + "/test", exist_ok=True)

if DATASET_DIR_IN_DRIVE:
    for rel in FILE_IDS:
        srcp = os.path.join(DATASET_DIR_IN_DRIVE, rel)
        dstp = os.path.join(DATA_DIR, rel)
        if not os.path.exists(dstp):
            print("copy", srcp)
            shutil.copy(srcp, dstp)
else:
    import gdown
    for rel, fid in FILE_IDS.items():
        dstp = os.path.join(DATA_DIR, rel)
        if os.path.exists(dstp) and os.path.getsize(dstp) == EXPECTED[rel]:
            print("have", rel)
            continue
        print("downloading", rel, "...")
        gdown.download(id=fid, output=dstp, quiet=False)

for rel in FILE_IDS:
    p = os.path.join(DATA_DIR, rel)
    ok = os.path.exists(p) and os.path.getsize(p) == EXPECTED[rel]
    print(("OK  " if ok else "BAD "), rel,
          os.path.getsize(p) if os.path.exists(p) else "missing")


## STEP 3 — Write the pipeline source code
The next three cells write the exact same code that ships in the
submission package (`code/business_entity_resolution/`).


In [ ]:
%%writefile src/ber_core.py
"""
Business Entity Resolution — core pipeline (blocking + features + LightGBM matcher).

Memory-efficient, country-partitioned design:
  1. Stream the source TSVs and split them into per-country part files.
  2. Per country: build an inverted blocking index over Source-2/3 records,
     query every Source-1 record, keep the top-K candidates (this exact set is
     what goes into candidate_pairs.tsv).
  3. Score every candidate pair with RapidFuzz similarity features and a
     LightGBM classifier; accept pairs above a threshold tuned for macro F_0.5.

Only pure-algorithm libraries are used (pandas / numpy / rapidfuzz / scikit-learn
/ lightgbm) — no external data, gazetteers, or lookup services.  LightGBM is a
gradient-boosted tree model (far under the 8B-parameter cap, MIT licensed).
"""

from __future__ import annotations

import csv
import os
import re
import sys
import time
import unicodedata
from array import array
from collections import defaultdict
from typing import Dict, Iterable, List, Sequence, Set, Tuple

import numpy as np
from rapidfuzz import fuzz

csv.field_size_limit(10_000_000)

# ---------------------------------------------------------------------------
# Normalisation (small hand-written dictionaries only — allowed by the rules)
# ---------------------------------------------------------------------------

LEGAL_SUFFIXES = {
    "inc", "incorporated", "corp", "corporation", "llc", "llp", "lp", "plc",
    "ltd", "limited", "pvt", "private", "co", "company", "pc", "pllc",
    "sarl", "sasu", "sas", "eurl", "sci", "snc", "gie", "cie",
    "gmbh", "ag", "sa", "lc",
}

NAME_STOPWORDS = {
    "the", "and", "of", "in", "at", "by", "for", "with", "a", "an",
    "et", "de", "du", "des", "la", "le", "les", "l", "d",
}

ABBREV = {
    # streets / addresses
    "st": "street", "rd": "road", "ave": "avenue", "av": "avenue",
    "dr": "drive", "ln": "lane", "blvd": "boulevard", "bd": "boulevard",
    "hwy": "highway", "ct": "court", "pl": "place", "sq": "square",
    "apt": "apartment", "ste": "suite", "fl": "floor", "flr": "floor",
    "bldg": "building", "opp": "opposite", "nr": "near",
    "n": "north", "s": "south", "e": "east", "w": "west",
}

ADDR_STOPWORDS = {
    "street", "road", "avenue", "drive", "lane", "boulevard", "court",
    "place", "way", "highway", "square", "near", "opposite", "behind",
    "beside", "next", "floor", "unit", "apartment", "suite", "building",
    "block", "tower", "house", "flat", "shop", "plot", "sector", "phase",
    "no", "number", "ho", "off",
    "nagar", "colony", "marg", "gali", "chowk", "vihar", "pura", "puram",
    "rue", "chemin", "impasse", "allee", "route", "bis",
}

_PUNCT_RE = re.compile(r"[^\w\s]")
_WS_RE = re.compile(r"\s+")
_NUM_RE = re.compile(r"\d{1,8}")
_NONALNUM_RE = re.compile(r"[^a-z0-9]")


def normalize_text(text: str) -> str:
    """Unicode-fold, lowercase, strip accents/punctuation, collapse spaces."""
    if not text:
        return ""
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = text.lower()
    text = text.replace("&", " and ")
    text = _PUNCT_RE.sub(" ", text)
    return _WS_RE.sub(" ", text).strip()


def name_core_tokens(norm_name: str) -> List[str]:
    """Significant name tokens: drop legal suffixes / stopwords / bare digits."""
    out = []
    for w in norm_name.split():
        if w in LEGAL_SUFFIXES or w in NAME_STOPWORDS:
            continue
        out.append(w)
    return out


def addr_tokens(norm_addr: str) -> List[str]:
    out = []
    for w in norm_addr.split():
        w = ABBREV.get(w, w)
        if w in ADDR_STOPWORDS or len(w) <= 1:
            continue
        out.append(w)
    return out


def compress(norm: str) -> str:
    return _NONALNUM_RE.sub("", norm)


def numbers_of(text: str) -> List[str]:
    return _NUM_RE.findall(text or "")


# ---------------------------------------------------------------------------
# Record container (memory-light: plain tuple layout)
# ---------------------------------------------------------------------------
# record = (entity_id, norm_name, norm_addr)


def read_source_tsv(path: str) -> Iterable[Tuple[str, str, str, str]]:
    """Yield (entity_id, business_name, business_address, country)."""
    with open(path, "r", encoding="utf-8", newline="") as f:
        reader = csv.reader(f, delimiter="\t")
        header = next(reader, None)
        for row in reader:
            if not row:
                continue
            eid = row[0].strip()
            name = row[1] if len(row) > 1 else ""
            addr = row[2] if len(row) > 2 else ""
            country = (row[3] if len(row) > 3 else "").strip()
            yield eid, name, addr, country


def split_by_country(src_paths: Dict[str, str], out_dir: str, tag: str) -> Dict[str, Dict[str, str]]:
    """Split each source file into per-country part files.

    Returns {country_key: {source_label: part_path}}.
    Country is treated as an open set of string labels.
    """
    os.makedirs(out_dir, exist_ok=True)
    handles: Dict[Tuple[str, str], object] = {}
    result: Dict[str, Dict[str, str]] = defaultdict(dict)

    def key_of(country: str) -> str:
        k = re.sub(r"[^A-Za-z0-9]+", "_", country.strip()) or "UNKNOWN"
        return k

    try:
        for label, path in src_paths.items():
            for eid, name, addr, country in read_source_tsv(path):
                ck = key_of(country)
                hk = (ck, label)
                if hk not in handles:
                    part = os.path.join(out_dir, f"{tag}_{label}_{ck}.tsv")
                    handles[hk] = open(part, "w", encoding="utf-8")
                    result[ck][label] = part
                handles[hk].write(f"{eid}\t{name}\t{addr}\n")
    finally:
        for h in handles.values():
            h.close()
    return dict(result)


def read_part(path: str) -> Iterable[Tuple[str, str, str]]:
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 3:
                yield parts[0], parts[1], parts[2]
            elif len(parts) == 2:
                yield parts[0], parts[1], ""


# ---------------------------------------------------------------------------
# Blocking
# ---------------------------------------------------------------------------

class Blocker:
    """Inverted-index blocking over Source-2/3 records of one country.

    Keys per record:
      * full compressed name             (weight 4)
      * compressed-name 8-char prefix    (weight 3)
      * first two core name tokens       (weight 3)
      * each core name token             (weight ~idf, capped postings)
      * rarest name token + addr number  (weight 2)
    Candidates are ranked by accumulated key weight; top-K survive.

    Implementation note: postings are kept as one flat (key-hash, record-idx)
    pair list that is sorted once after loading (``finalize``).  Lookups use
    binary search.  Record strings live in a single byte blob with offset
    arrays.  This keeps memory bounded (~8 bytes/posting, zero per-string
    overhead) so millions of records fit in well under a GB.
    """

    MAX_POSTING = 400          # ignore keys more frequent than this at query time

    def __init__(self):
        self._kh = array("I")   # 32-bit key hashes
        self._ki = array("i")   # record indices
        self.sorted_h = None
        self.sorted_i = None
        # record storage: [id bytes | name bytes | addr bytes] per record
        self._blob = bytearray()
        self._offs = array("q", [0])
        self.n_records = 0

    @staticmethod
    def _h32(key: str) -> int:
        return hash(key) & 0xFFFFFFFF

    def get(self, i: int) -> Tuple[str, str, str]:
        """Return (entity_id, norm_name, norm_addr) for record i."""
        o = self._offs
        b = self._blob
        base = 3 * i
        return (
            bytes(b[o[base]:o[base + 1]]).decode("utf-8"),
            bytes(b[o[base + 1]:o[base + 2]]).decode("utf-8"),
            bytes(b[o[base + 2]:o[base + 3]]).decode("utf-8"),
        )

    # -- shared key generation ------------------------------------------------
    @staticmethod
    def keys_for(norm_name: str, norm_addr: str) -> List[Tuple[str, float]]:
        toks = name_core_tokens(norm_name)
        comp = compress(" ".join(toks) if toks else norm_name)
        keys: List[Tuple[str, float]] = []
        if comp:
            keys.append(("C:" + comp, 4.0))
            if len(comp) >= 8:
                keys.append(("P:" + comp[:8], 3.0))
            elif len(comp) >= 5:
                keys.append(("P:" + comp[:5], 2.0))
        if len(toks) >= 2:
            keys.append(("T2:" + toks[0] + "_" + toks[1], 3.0))
        for t in toks[:6]:
            if len(t) >= 3:
                keys.append(("T:" + t, 1.0))
        nums = numbers_of(norm_addr)
        if toks and nums:
            keys.append(("NA:" + toks[0] + "_" + nums[0], 2.0))
        return keys

    def add(self, eid: str, norm_name: str, norm_addr: str) -> None:
        idx = self.n_records
        self.n_records += 1
        blob, offs = self._blob, self._offs
        blob += eid.encode("utf-8")
        offs.append(len(blob))
        blob += norm_name.encode("utf-8")
        offs.append(len(blob))
        blob += norm_addr.encode("utf-8")
        offs.append(len(blob))
        kh, ki = self._kh, self._ki
        h32 = self._h32
        for key, _w in self.keys_for(norm_name, norm_addr):
            kh.append(h32(key))
            ki.append(idx)

    def finalize(self) -> None:
        """Sort the flat posting list once so queries can binary-search it."""
        kh = np.asarray(self._kh, dtype=np.uint32)
        self._kh = array("I")
        ki = np.asarray(self._ki, dtype=np.int32)
        self._ki = array("i")
        order = np.argsort(kh, kind="stable")
        self.sorted_h = kh[order]
        kh = None
        self.sorted_i = ki[order]
        del ki, order

    def query(self, norm_name: str, norm_addr: str, top_k: int) -> List[Tuple[int, float]]:
        if self.sorted_h is None:
            self.finalize()
        scores: Dict[int, float] = defaultdict(float)
        sh, si = self.sorted_h, self.sorted_i
        h32 = self._h32
        for key, w in self.keys_for(norm_name, norm_addr):
            h = h32(key)
            lo = np.searchsorted(sh, h, "left")
            hi = np.searchsorted(sh, h, "right")
            n = int(hi - lo)
            if n == 0 or n > self.MAX_POSTING:
                continue
            # frequency-aware weight: rare keys count more
            kw = w * (1.0 + 2.0 / (1.0 + 0.05 * n))
            for idx in si[lo:hi]:
                scores[int(idx)] += kw
        if not scores:
            return []
        items = sorted(scores.items(), key=lambda kv: -kv[1])[:top_k]
        # prune weak tail candidates: keeps the candidate set small without
        # hurting recall (they virtually never survive the matcher anyway)
        best = items[0][1]
        floor = max(2.0, 0.30 * best)
        return [it for it in items if it[1] >= floor]


# ---------------------------------------------------------------------------
# Pairwise features
# ---------------------------------------------------------------------------

FEATURE_NAMES = [
    "name_ratio", "name_token_sort", "name_token_set", "name_partial",
    "name_jaccard", "comp_ratio", "comp_prefix", "name_len_diff",
    "addr_ratio", "addr_token_set", "addr_token_sort", "addr_jaccard",
    "num_jaccard", "num_first_match", "addr_missing",
    "block_score", "block_rank", "is_source2",
]


def jaccard(a: Set[str], b: Set[str]) -> float:
    if not a or not b:
        return 0.0
    inter = len(a & b)
    if inter == 0:
        return 0.0
    return inter / (len(a) + len(b) - inter)


def pair_features(
    s1_name: str, s1_addr: str,
    c_name: str, c_addr: str,
    block_score: float, block_rank: int, is_s2: bool,
    _cache: dict = None,
) -> List[float]:
    if _cache is not None and "s1" in _cache:
        n1toks, a1toks, nums1, comp1 = _cache["s1"]
    else:
        n1toks = set(name_core_tokens(s1_name))
        a1toks = set(addr_tokens(s1_addr))
        nums1 = set(numbers_of(s1_addr))
        comp1 = compress(s1_name)
        if _cache is not None:
            _cache["s1"] = (n1toks, a1toks, nums1, comp1)

    n2toks = set(name_core_tokens(c_name))
    a2toks = set(addr_tokens(c_addr))
    nums2 = set(numbers_of(c_addr))
    comp2 = compress(c_name)

    name_ratio = fuzz.ratio(s1_name, c_name) / 100.0
    name_tsort = fuzz.token_sort_ratio(s1_name, c_name) / 100.0
    name_tset = fuzz.token_set_ratio(s1_name, c_name) / 100.0
    name_part = fuzz.partial_ratio(s1_name, c_name) / 100.0
    name_jac = jaccard(n1toks, n2toks)
    comp_ratio = fuzz.ratio(comp1, comp2) / 100.0
    pref = 0
    for x, y in zip(comp1, comp2):
        if x != y:
            break
        pref += 1
    comp_prefix = pref / max(1, min(len(comp1), len(comp2)))
    l1, l2 = len(s1_name), len(c_name)
    name_len_diff = abs(l1 - l2) / max(1, l1, l2)

    addr_missing = 1.0 if (not c_addr or not s1_addr) else 0.0
    if addr_missing:
        addr_ratio = addr_tset = addr_tsort = addr_jac = 0.0
    else:
        addr_ratio = fuzz.ratio(s1_addr, c_addr) / 100.0
        addr_tset = fuzz.token_set_ratio(s1_addr, c_addr) / 100.0
        addr_tsort = fuzz.token_sort_ratio(s1_addr, c_addr) / 100.0
        addr_jac = jaccard(a1toks, a2toks)

    num_jac = jaccard(nums1, nums2)
    n1f = next(iter(numbers_of(s1_addr)), None)
    n2f = next(iter(numbers_of(c_addr)), None)
    num_first = 1.0 if (n1f is not None and n1f == n2f) else 0.0

    return [
        name_ratio, name_tsort, name_tset, name_part,
        name_jac, comp_ratio, comp_prefix, name_len_diff,
        addr_ratio, addr_tset, addr_tsort, addr_jac,
        num_jac, num_first, addr_missing,
        block_score, float(block_rank), 1.0 if is_s2 else 0.0,
    ]


# ---------------------------------------------------------------------------
# F_0.5 utilities
# ---------------------------------------------------------------------------

def f05(pred: Set[str], truth: Set[str]) -> float:
    if not truth and not pred:
        return 1.0
    if not pred or not truth:
        return 0.0
    tp = len(pred & truth)
    if tp == 0:
        return 0.0
    p = tp / len(pred)
    r = tp / len(truth)
    return 1.25 * p * r / (0.25 * p + r)


def macro_f05(preds: Dict[str, Set[str]], truths: Dict[str, Set[str]]) -> float:
    if not truths:
        return 0.0
    return sum(f05(preds.get(k, set()), v) for k, v in truths.items()) / len(truths)


In [ ]:
%%writefile src/pipeline.py
"""
End-to-end Business Entity Resolution pipeline.

Usage:
    python -m src.pipeline --data-dir dataset --out-dir output \
        --work-dir work --model-path models/matcher_lgbm.txt \
        [--train-sample 150000] [--top-k 12] [--skip-train]

Stages:
  1. TRAIN  : country-split train data -> blocking -> labelled pairs ->
              LightGBM -> threshold tuned for macro F_0.5 on a held-out split.
  2. PREDICT: country-split test data -> blocking (top-K candidates per S1) ->
              model scores -> matching_results.tsv + candidate_pairs.tsv.
"""

from __future__ import annotations

import argparse
import csv
import json
import os
import random
import sys
import time
from collections import defaultdict
from typing import Dict, List, Set, Tuple

import numpy as np
import lightgbm as lgb

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from src.ber_core import (
    Blocker, FEATURE_NAMES, f05, macro_f05, normalize_text,
    pair_features, read_part, read_source_tsv, split_by_country,
)

RANDOM_SEED = 42


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def load_ground_truth(path: str, only: Set[str] = None) -> Dict[str, Set[str]]:
    """Load ground truth; when ``only`` is given keep just those S1 entities
    (memory saver: the full file holds >2M rows)."""
    gt: Dict[str, Set[str]] = {}
    with open(path, "r", encoding="utf-8") as f:
        header = f.readline()
        for line in f:
            s1, _, rest = line.rstrip("\n").partition("\t")
            if only is not None and s1 not in only:
                continue
            ids = set(x for x in rest.split(",") if x) if rest.strip() else set()
            gt[s1] = ids
    return gt


def build_index_for_country(parts: Dict[str, str]) -> Blocker:
    """Index all Source-2/3 records of one country."""
    blocker = Blocker()
    for label in ("s2", "s3"):
        p = parts.get(label)
        if not p:
            continue
        for eid, name, addr in read_part(p):
            blocker.add(eid, normalize_text(name), normalize_text(addr))
    blocker.finalize()
    return blocker


def candidates_for_s1(
    blocker: Blocker, name: str, addr: str, top_k: int
) -> List[Tuple[str, str, str, float, int]]:
    """Return [(cand_id, cand_norm_name, cand_norm_addr, block_score, rank)]."""
    nn, na = normalize_text(name), normalize_text(addr)
    hits = blocker.query(nn, na, top_k)
    out = []
    for rank, (idx, score) in enumerate(hits):
        cid, cname, caddr = blocker.get(idx)
        out.append((cid, cname, caddr, score, rank))
    return out


# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------

def run_training(
    data_dir: str,
    work_dir: str,
    model_path: str,
    train_sample: int,
    top_k: int,
    val_frac: float = 0.2,
) -> Tuple[lgb.Booster, float]:
    t0 = time.time()
    rng = random.Random(RANDOM_SEED)
    print("[train] splitting train sources by country ...", flush=True)
    parts = split_by_country(
        {
            "s1": os.path.join(data_dir, "train", "train_source1.tsv"),
            "s2": os.path.join(data_dir, "train", "train_source2.tsv"),
            "s3": os.path.join(data_dir, "train", "train_source3.tsv"),
        },
        os.path.join(work_dir, "parts_train"),
        "train",
    )
    # Sample S1 entities proportionally per country (streaming, memory-light)
    s1_counts: Dict[str, int] = {}
    for ck, p in parts.items():
        if "s1" not in p:
            continue
        with open(p["s1"], "r", encoding="utf-8") as f:
            s1_counts[ck] = sum(1 for _ in f)
    total_s1 = sum(s1_counts.values())
    frac = min(1.0, train_sample / max(1, total_s1))
    print(f"[train] S1 total {total_s1:,}; sampling ~{frac:.1%}", flush=True)

    # Pre-select the sampled entities so the ground truth can be loaded
    # filtered (the full GT map holds >2M rows and wastes ~1 GB otherwise).
    sampled_by_country: Dict[str, List[Tuple[str, str, str]]] = {}
    sampled_ids: Set[str] = set()
    for ck in s1_counts:
        recs = [r for r in read_part(parts[ck]["s1"]) if rng.random() < frac]
        sampled_by_country[ck] = recs
        sampled_ids.update(r[0] for r in recs)
    gt = load_ground_truth(
        os.path.join(data_dir, "train", "train_ground_truth.tsv"), only=sampled_ids)
    print(f"[train] ground truth rows kept: {len(gt):,}", flush=True)

    X_chunks: List[np.ndarray] = []
    y_rows: List[int] = []
    pair_meta: List[Tuple[str, str]] = []   # (s1_id, cand_id)
    entity_split: Dict[str, int] = {}       # s1_id -> 0 train / 1 val
    truth_by_entity: Dict[str, Set[str]] = {}
    X_buf: List[List[float]] = []

    def _flush_X():
        if X_buf:
            X_chunks.append(np.asarray(X_buf, dtype=np.float32))
            X_buf.clear()

    for ck in sorted(s1_counts, key=lambda c: s1_counts[c]):
        sampled = sampled_by_country.pop(ck, [])
        if not sampled:
            continue
        print(f"[train] country={ck}: S1={s1_counts[ck]:,} sampled={len(sampled):,} "
              f"— building index ...", flush=True)
        blocker = build_index_for_country(parts[ck])
        print(f"[train]   index size: {blocker.n_records:,} records, "
              f"{len(blocker.sorted_h):,} postings", flush=True)
        for i, (s1_id, name, addr) in enumerate(sampled):
            truth = gt.get(s1_id, set())
            cands = candidates_for_s1(blocker, name, addr, top_k)
            split = 1 if rng.random() < val_frac else 0
            entity_split[s1_id] = split
            truth_by_entity[s1_id] = truth
            nn, na = normalize_text(name), normalize_text(addr)
            cache: dict = {}
            for cid, cname, caddr, bscore, brank in cands:
                feats = pair_features(nn, na, cname, caddr, bscore, brank,
                                      cid.startswith("S2-"), cache)
                X_buf.append(feats)
                y_rows.append(1 if cid in truth else 0)
                pair_meta.append((s1_id, cid))
            if len(X_buf) >= 100_000:
                _flush_X()
            if (i + 1) % 20000 == 0:
                print(f"[train]   {i+1:,}/{len(sampled):,} queried "
                      f"({time.time()-t0:,.0f}s)", flush=True)
        del blocker

    _flush_X()
    X = np.concatenate(X_chunks, axis=0) if X_chunks else np.zeros((0, len(FEATURE_NAMES)), np.float32)
    X_chunks.clear()
    y = np.asarray(y_rows, dtype=np.int8)
    is_val = np.asarray([entity_split[m[0]] for m in pair_meta], dtype=np.int8)
    print(f"[train] pairs: {len(y):,} (pos {int(y.sum()):,}) "
          f"val pairs {int((is_val==1).sum()):,}", flush=True)

    dtrain = lgb.Dataset(X[is_val == 0], label=y[is_val == 0],
                         feature_name=FEATURE_NAMES)
    dval = lgb.Dataset(X[is_val == 1], label=y[is_val == 1],
                       reference=dtrain, feature_name=FEATURE_NAMES)
    params = dict(
        objective="binary", metric="auc", learning_rate=0.07,
        num_leaves=63, min_data_in_leaf=40, feature_fraction=0.9,
        bagging_fraction=0.9, bagging_freq=1, seed=RANDOM_SEED, verbosity=-1,
        num_threads=os.cpu_count() or 2,
    )
    booster = lgb.train(
        params, dtrain, num_boost_round=600,
        valid_sets=[dval], valid_names=["val"],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(100)],
    )
    print(f"[train] best iteration: {booster.best_iteration}", flush=True)

    # ---- threshold tuning on validation entities (macro F_0.5, incl. singletons)
    val_idx = np.where(is_val == 1)[0]
    probs = booster.predict(X[val_idx], num_iteration=booster.best_iteration)
    by_entity: Dict[str, List[Tuple[str, float]]] = defaultdict(list)
    for j, k in enumerate(val_idx):
        s1_id, cid = pair_meta[k]
        by_entity[s1_id].append((cid, float(probs[j])))
    val_truth = {e: t for e, t in truth_by_entity.items() if entity_split[e] == 1}

    best_t, best_score = 0.5, -1.0
    for t in np.arange(0.20, 0.96, 0.02):
        preds = {
            e: {cid for cid, p in by_entity.get(e, []) if p >= t}
            for e in val_truth
        }
        s = macro_f05(preds, val_truth)
        if s > best_score:
            best_score, best_t = s, float(t)
    print(f"[train] tuned threshold={best_t:.2f}  val macro-F0.5={best_score:.4f}",
          flush=True)

    os.makedirs(os.path.dirname(model_path) or ".", exist_ok=True)
    booster.save_model(model_path, num_iteration=booster.best_iteration)
    with open(model_path + ".meta.json", "w") as f:
        json.dump({"threshold": best_t, "val_macro_f05": best_score,
                   "top_k": top_k, "features": FEATURE_NAMES,
                   "best_iteration": booster.best_iteration}, f, indent=2)
    print(f"[train] model saved -> {model_path} ({time.time()-t0:,.0f}s total)",
          flush=True)
    return booster, best_t


# ---------------------------------------------------------------------------
# Inference
# ---------------------------------------------------------------------------

def run_inference(
    data_dir: str,
    work_dir: str,
    out_dir: str,
    model_path: str,
    top_k: int,
    batch_pairs: int = 200_000,
) -> None:
    t0 = time.time()
    booster = lgb.Booster(model_file=model_path)
    with open(model_path + ".meta.json") as f:
        meta = json.load(f)
    threshold = meta["threshold"]
    print(f"[predict] threshold={threshold:.2f} top_k={top_k}", flush=True)

    print("[predict] splitting test sources by country ...", flush=True)
    parts = split_by_country(
        {
            "s1": os.path.join(data_dir, "test", "test_source1.tsv"),
            "s2": os.path.join(data_dir, "test", "test_source2.tsv"),
            "s3": os.path.join(data_dir, "test", "test_source3.tsv"),
        },
        os.path.join(work_dir, "parts_test"),
        "test",
    )

    os.makedirs(out_dir, exist_ok=True)
    match_path = os.path.join(out_dir, "matching_results.tsv")
    cand_path = os.path.join(out_dir, "candidate_pairs.tsv")
    n_entities = n_matched = n_cand_total = 0

    with open(match_path, "w", encoding="utf-8") as fm, \
         open(cand_path, "w", encoding="utf-8") as fc:
        fm.write("source1_entity_id\tmatched_entity_ids\n")
        fc.write("source1_entity_id\tcandidate_entity_ids\n")

        for ck in sorted(parts, key=lambda c: os.path.getsize(parts[c].get("s1", parts[c].get("s2", "/dev/null"))) if parts[c] else 0):
            p = parts[ck]
            if "s1" not in p:
                continue
            print(f"[predict] country={ck}: building index ...", flush=True)
            blocker = build_index_for_country(p)
            print(f"[predict]   index: {blocker.n_records:,} records", flush=True)

            # batched scoring to keep memory flat
            buf_feats: List[List[float]] = []
            buf_meta: List[Tuple[str, str]] = []
            buf_entities: List[Tuple[str, List[str]]] = []  # (s1_id, cand_ids)

            def flush():
                nonlocal n_matched
                if not buf_entities:
                    return
                if buf_feats:
                    probs = booster.predict(
                        np.asarray(buf_feats, dtype=np.float32),
                        num_iteration=booster.best_iteration or None)
                else:
                    probs = np.zeros(0)
                pos = 0
                by_entity: Dict[str, List[Tuple[str, float]]] = {}
                for (s1_id, cid) in buf_meta:
                    by_entity.setdefault(s1_id, []).append((cid, float(probs[pos])))
                    pos += 1
                for s1_id, cand_ids in buf_entities:
                    scored = by_entity.get(s1_id, [])
                    matched = [cid for cid, pr in scored if pr >= threshold]
                    n_matched += len(matched)
                    fm.write(f"{s1_id}\t{','.join(matched)}\n")
                    fc.write(f"{s1_id}\t{','.join(cand_ids)}\n")
                buf_feats.clear(); buf_meta.clear(); buf_entities.clear()

            count = 0
            for s1_id, name, addr in read_part(p["s1"]):
                nn, na = normalize_text(name), normalize_text(addr)
                hits = blocker.query(nn, na, top_k)
                cand_ids = []
                cache: dict = {}
                for rank, (idx, score) in enumerate(hits):
                    cid, cname, caddr = blocker.get(idx)
                    cand_ids.append(cid)
                    buf_feats.append(pair_features(
                        nn, na, cname, caddr,
                        score, rank, cid.startswith("S2-"), cache))
                    buf_meta.append((s1_id, cid))
                buf_entities.append((s1_id, cand_ids))
                n_entities += 1
                n_cand_total += len(cand_ids)
                count += 1
                if len(buf_feats) >= batch_pairs:
                    flush()
                if count % 100000 == 0:
                    print(f"[predict]   {count:,} S1 done "
                          f"({time.time()-t0:,.0f}s)", flush=True)
            flush()
            del blocker
            print(f"[predict] country={ck} done ({time.time()-t0:,.0f}s)", flush=True)

    print(f"[predict] wrote {match_path} and {cand_path}", flush=True)
    print(f"[predict] entities={n_entities:,} matches={n_matched:,} "
          f"avg candidates/entity={n_cand_total/max(1,n_entities):.2f}", flush=True)


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

def main():
    ap = argparse.ArgumentParser(description="Business Entity Resolution pipeline")
    ap.add_argument("--data-dir", required=True,
                    help="folder containing train/ and test/ subfolders")
    ap.add_argument("--out-dir", default="output")
    ap.add_argument("--work-dir", default="work")
    ap.add_argument("--model-path", default="models/matcher_lgbm.txt")
    ap.add_argument("--train-sample", type=int, default=150_000,
                    help="number of S1 training entities to sample")
    ap.add_argument("--top-k", type=int, default=12,
                    help="candidates kept per S1 entity (blocking output)")
    ap.add_argument("--skip-train", action="store_true",
                    help="reuse an existing model file")
    ap.add_argument("--skip-predict", action="store_true")
    args = ap.parse_args()

    os.makedirs(args.work_dir, exist_ok=True)
    if not args.skip_train:
        run_training(args.data_dir, args.work_dir, args.model_path,
                     args.train_sample, args.top_k)
    if not args.skip_predict:
        run_inference(args.data_dir, args.work_dir, args.out_dir,
                      args.model_path, args.top_k)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile utils/validate_submission.py
#!/usr/bin/env python3
"""
ML Challenge 2026 — Submission Validator (stdlib only, Python 3.8+).

Run from the project directory:

    python3 utils/validate_submission.py \
        --matching output/matching_results.tsv \
        --candidate output/candidate_pairs.tsv \
        --test-dir dataset/test

Exit code 0 = safe to submit; 1 = fix the listed issues.
"""

import argparse
import os
import sys

DELIM = "\t"
MAX_EXAMPLES = 5
MATCHING_HEADER = ["source1_entity_id", "matched_entity_ids"]
CANDIDATE_HEADER = ["source1_entity_id", "candidate_entity_ids"]


def read_ids(path):
    with open(path, encoding="utf-8") as f:
        next(f, None)
        return {line.split(DELIM, 1)[0].strip() for line in f if line.strip()}


def examples(items):
    items = sorted(items)
    shown = ", ".join(items[:MAX_EXAMPLES])
    if len(items) > MAX_EXAMPLES:
        return f"{len(items)} total, e.g. {shown}, ..."
    return shown


def load_match_targets(test_dir, warnings):
    targets = set()
    for name in ("test_source2.tsv", "test_source3.tsv"):
        path = os.path.join(test_dir, name)
        if not os.path.isfile(path):
            warnings.append(
                f"{path} not found — skipping the (optional) check that matched "
                f"IDs exist in the test set."
            )
            return None
        targets |= read_ids(path)
    return targets


def validate_id_list_file(path, expected_header, col_label, required, valid_ids, errors):
    if not os.path.isfile(path):
        errors.append(f"File not found: {path}")
        return None

    name = os.path.basename(path)
    mapping = {}
    seen, dup_rows, intra_dupes = set(), set(), set()
    self_matches, wrong_prefix, unknown = set(), set(), set()
    n_rows = empties = 0

    with open(path, encoding="utf-8") as f:
        header = f.readline()
        if not header:
            errors.append(f"{name} is empty.")
            return None
        if DELIM not in header and "," in header:
            errors.append(
                f"{name}: header has no TAB but contains commas — the file looks "
                "COMMA-separated. Submissions must be TAB-separated (.tsv)."
            )
            return None
        cols = [c.strip().lower() for c in header.rstrip("\n").split(DELIM)]
        if cols != expected_header:
            errors.append(
                f"{name}: unexpected header {cols}. "
                f"Expected exactly {expected_header} (tab-separated)."
            )
            return None

        for line_num, line in enumerate(f, start=2):
            s1, tab, rest = line.partition(DELIM)
            if not tab:
                if s1.strip():
                    errors.append(
                        f"{name}: malformed row (no tab) at line {line_num}: "
                        f"{line.rstrip()!r}"
                    )
                continue

            n_rows += 1
            if s1 in seen:
                dup_rows.add(s1)
            seen.add(s1)

            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            if not ids:
                empties += 1
                mapping[s1] = set()
                continue
            if len(ids) != len(set(ids)):
                intra_dupes.add(s1)
            id_set = set(ids)
            mapping[s1] = id_set
            for mid in id_set:
                if mid.startswith("S1-"):
                    self_matches.add(mid)
                elif not mid.startswith(("S2-", "S3-")):
                    wrong_prefix.add(mid)
                elif valid_ids is not None and mid not in valid_ids:
                    unknown.add(mid)

    findings = [
        (dup_rows, "{name}: duplicate source1_entity_id row(s): {ex}."),
        (intra_dupes, "{name}: repeated ID inside a {col} list for: {ex}."),
        (self_matches, "{name}: {col} contains Source-1 IDs (self-matches): {ex}."),
        (wrong_prefix, "{name}: {col} contains IDs without an S2-/S3- prefix: {ex}."),
        (unknown, "{name}: {col} references IDs not in the test Source-2/3 files: {ex}."),
        (required - seen, "{name}: required S1 entity(ies) missing: {ex}."),
        (seen - required, "{name}: row(s) using an S1 ID that is not in the test set: {ex}."),
    ]
    for offenders, message in findings:
        if offenders:
            errors.append(message.format(name=name, ex=examples(offenders), col=col_label))

    print(f"  {name}: {n_rows} rows ({empties} empty, {n_rows - empties} non-empty).")
    return mapping


def validate(matching_path, candidate_path, test_dir, check_ids=False):
    errors, warnings = [], []

    source1 = os.path.join(test_dir, "test_source1.tsv")
    if not os.path.isfile(source1):
        errors.append(f"Test source1 file not found: {source1} (check --test-dir).")
        return errors, warnings
    required = read_ids(source1)
    print(f"  required S1 entities: {len(required)}")

    if check_ids:
        valid_ids = load_match_targets(test_dir, warnings)
        if valid_ids is not None:
            print(f"  valid S2/S3 match IDs: {len(valid_ids)}")
    else:
        valid_ids = None
        warnings.append(
            "ID-existence check is OFF (the default). Re-run with --check-ids to "
            "enable it (needs test_source2/3.tsv; uses more memory)."
        )

    matched = validate_id_list_file(
        matching_path, MATCHING_HEADER, "matched_entity_ids", required, valid_ids, errors
    )

    candidate = None
    if candidate_path and os.path.isfile(candidate_path):
        candidate = validate_id_list_file(
            candidate_path, CANDIDATE_HEADER, "candidate_entity_ids",
            required, valid_ids, errors,
        )
    elif candidate_path:
        warnings.append(
            f"{candidate_path} not found — skipping candidate_pairs.tsv checks."
        )

    if matched is not None and candidate is not None:
        offenders = {
            s1 for s1, mids in matched.items() if mids - candidate.get(s1, set())
        }
        if offenders:
            warnings.append(
                f"{len(offenders)} S1 entity(ies) have matched IDs not present in "
                f"candidate_pairs.tsv, e.g. {examples(offenders)}."
            )

    return errors, warnings


def main():
    parser = argparse.ArgumentParser(
        description="Validate submission output files before submitting."
    )
    parser.add_argument("--matching", "-m", default="output/matching_results.tsv")
    parser.add_argument("--candidate", "-c", default=None)
    parser.add_argument("--test-dir", "-t", default="dataset/test")
    parser.add_argument("--check-ids", action="store_true")
    args = parser.parse_args()

    candidate_path = args.candidate or "output/candidate_pairs.tsv"

    print("ML Challenge 2026 — submission validator")
    print(f"  test dir: {args.test_dir}")
    try:
        errors, warnings = validate(
            args.matching, candidate_path, args.test_dir, check_ids=args.check_ids
        )
    except UnicodeDecodeError:
        print("\nFAIL — a file is not valid UTF-8 tab-separated text.")
        return 1
    except OSError as exc:
        print(f"\nFAIL — could not read a file: {exc}.")
        return 1

    print()
    for warning in warnings:
        print(f"WARNING: {warning}")
    if errors:
        print(f"FAIL — {len(errors)} issue(s) to fix before submitting:")
        for i, error in enumerate(errors, 1):
            print(f"  {i}. {error}")
        return 1
    print("PASS — no blocking issues found. Safe to submit.")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
open('src/__init__.py','w').close()
print('src package ready')


In [ ]:
# ============================================================
# STEP 4 — Run the FULL pipeline (training + inference)
# On a standard Colab CPU runtime this takes roughly 1.5-3 hours.
# Progress is printed continuously. Re-run with --skip-train to reuse
# a trained model.
# ============================================================
!python -m src.pipeline \
    --data-dir /content/dataset \
    --out-dir  output \
    --work-dir work \
    --model-path models/matcher_lgbm.txt \
    --train-sample 150000 \
    --top-k 12


In [ ]:
# ============================================================
# STEP 5 — Validate the submission files (official validator rules)
# ============================================================
!python3 utils/validate_submission.py \
    --matching output/matching_results.tsv \
    --candidate output/candidate_pairs.tsv \
    --test-dir /content/dataset/test


In [ ]:
# Quick look at the outputs
!head -5 output/matching_results.tsv
!head -5 output/candidate_pairs.tsv
!wc -l output/matching_results.tsv output/candidate_pairs.tsv


In [ ]:
# ============================================================
# STEP 6 — Save results to YOUR Google Drive + build the submission zip
# Outputs are copied to OUTPUT_DRIVE_DIR (created if missing).
# ============================================================
import os, shutil, zipfile

OUTPUT_DRIVE_DIR = "/content/drive/MyDrive/BER_submission"

os.makedirs(OUTPUT_DRIVE_DIR, exist_ok=True)
for f in ["matching_results.tsv", "candidate_pairs.tsv"]:
    shutil.copy(f"output/{f}", os.path.join(OUTPUT_DRIVE_DIR, f))
shutil.copy("models/matcher_lgbm.txt", os.path.join(OUTPUT_DRIVE_DIR, "matcher_lgbm.txt"))
shutil.copy("models/matcher_lgbm.txt.meta.json", os.path.join(OUTPUT_DRIVE_DIR, "matcher_lgbm.txt.meta.json"))

# submission zip: output/ + code/ + documentation
zip_path = os.path.join(OUTPUT_DRIVE_DIR, "team_submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    z.write("output/matching_results.tsv", "output/matching_results.tsv")
    z.write("output/candidate_pairs.tsv", "output/candidate_pairs.tsv")
    z.write("src/ber_core.py", "code/business_entity_resolution/src/ber_core.py")
    z.write("src/pipeline.py", "code/business_entity_resolution/src/pipeline.py")
    z.write("utils/validate_submission.py",
            "code/business_entity_resolution/utils/validate_submission.py")
    reqs = "numpy==1.26.4\npandas==2.2.2\nscikit-learn==1.5.1\nlightgbm==4.5.0\nrapidfuzz==3.9.6\ntqdm==4.66.4\n"
    z.writestr("code/business_entity_resolution/requirements.txt", reqs)
print("Saved to", OUTPUT_DRIVE_DIR)
print("Zip:", zip_path)
